# Build `Paragraph_Scrubbed_Window` for ablation experiment

**Purpose:** address Reviewer 3's concern that fact+principle gains may simply reflect richer citation-proximal text rather than doctrinal extraction.

**Experimental design:**

| Setup | Query input |
|---|---|
| Baseline | `Fact` |
| Existing | `Fact + Key Principle` |
| **NEW control** | `Fact + Paragraph_Scrubbed_Window` |

If `Fact + Principle` outperforms `Fact + Paragraph_Scrubbed_Window`, that means the LLM-distilled principle adds signal beyond raw citation-proximal text — refuting Reviewer 3's worry.

**This notebook does:**
1. For each of the 100,890 case-principle pairs, find the cited-case mention in the `Paragraph`
2. Take a ±200 word window around that anchor (~400 words = ~520 tokens; fits in BERT's 512 limit alongside the fact summary)
3. Scrub citation references to prevent leakage:
   - The cited-case name itself → `[CASE]`
   - Singapore neutral citations (`[2024] SGCA 5`) → `[CITATION]`
   - SLR/SLR(R)/MLJ reporters → `[CITATION]`
   - UK/Commonwealth citations (`[2020] AC`, `[2019] EWCA Civ 12`, etc.) → `[CITATION]`
   - Other `Party v Party` patterns → `[CASE]`
   - Pinpoint references (`at [60]`, `at paras 14-15`) → removed
4. Save as a new column `Paragraph_Scrubbed_Window` alongside the original data

**Output:** `COMBINED_ALL_CASES_FINAL_V2_with_scrubbed.csv` — same rows, with one extra column

## Configuration

In [1]:
import pandas as pd
import re
from pathlib import Path

# ----- Paths -----
INPUT_CSV  = "/Users/shannon/Desktop/V6 Method - Deepseek/Final Final Output/COMBINED_ALL_CASES_FINAL_V2.csv"

# Output to the same folder as this notebook
OUT_DIR = "/Users/shannon/Desktop/Scrubbed Paragraph Window"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = f"{OUT_DIR}/COMBINED_ALL_CASES_FINAL_V2_with_scrubbed.csv"

# ----- Window size -----
# 200 words each side = 400 words total = ~520 tokens after BPE
# Fact summary is ~45 words (~60 tokens). Together ~580 tokens, fits in 512-token encoders
# after [CLS] [SEP] overhead with mild truncation.
WORDS_EACH_SIDE = 200

print(f"Input:  {INPUT_CSV}")
print(f"Output: {OUTPUT_CSV}")
print(f"Window: ±{WORDS_EACH_SIDE} words around cited-case anchor")

Input:  /Users/shannon/Desktop/V6 Method - Deepseek/Final Final Output/COMBINED_ALL_CASES_FINAL_V2.csv
Output: /Users/shannon/Desktop/Scrubbed Paragraph Window/COMBINED_ALL_CASES_FINAL_V2_with_scrubbed.csv
Window: ±200 words around cited-case anchor


## Citation-scrubbing patterns

These regex patterns identify citations and case references that, if left in the text, would let the model retrieve via name-matching rather than substantive content (i.e. data leakage).

In [2]:
# ----- Singapore neutral citations -----
SG_NEUTRAL = re.compile(
    r'\[\s*\d{4}\s*\]\s*SG(?:CA|HC|HCF|HCR|CAI|HCI|HCA|DC|MC|FC|FB)(?:\([A-Z]\))?\s*\d+',
    re.IGNORECASE,
)

# ----- SLR / SLR(R) / MLJ / SAR / SCR with vol+page -----
SG_REPORTERS = re.compile(
    r'\[\s*\d{4}\s*\]\s*\d+\s*(?:SLR(?:\(R\))?|MLJ|SAR|SCR)\s*\d+',
    re.IGNORECASE,
)

# ----- UK / Commonwealth (with year prefix) -----
FOREIGN = re.compile(
    r'\[\s*\d{4}\s*\]\s*(?:\d+\s+)?'
    r'(?:AC|WLR|QB|KB|Ch|All\s*ER|EWCA(?:\s*Civ|\s*Crim)?|EWHC(?:\s*\([A-Za-z]+\))?|'
    r'UKSC|UKHL|UKPC|HCA|FCA|FCAFC|NSWCA|NSWSC|VSCA|VSC|NZSC|NZCA|NZHC|'
    r'HKCFA|HKCA|HKCFI|MLJ|BCLC|TLR|RPC|FSR|Lloyd\'?s\s*Rep)\s*\d*',
    re.IGNORECASE,
)

# ----- Older paren format -----
PAREN_REPORTERS = re.compile(
    r'\(\s*\d{4}\s*\)\s*\d+\s+'
    r'(?:CLR|LR|SLR|WLR|All\s*ER|HKLR|MLJ|TLR|RPC|FSR|BCLC|Lloyd\'?s\s*Rep|App\s*Cas|P\s*&\s*D)'
    r'\s*\d+',
    re.IGNORECASE,
)

# =====================================================================
# CASE-NAME PATTERNS — catch other cases mentioned in the paragraph
# =====================================================================
# Order: more specific FIRST, generic 'Party v Party' LAST.

# 1. PP v X / Public Prosecutor v X (Singapore criminal — 6.23%)
PP_V_PATTERN = re.compile(
    r'\b(?:Public\s+Prosecutor|PP)\s+v\.?\s+'
    r'[A-Z][\w\.\-\']+'
    r'(?:\s+(?:[A-Z][\w\.\-\']+|of|the|for|and|&|on|in|\([^)]*\)))*',
    re.UNICODE,
)

# 2. R v X (UK criminal — 1.36%)
R_V_PATTERN = re.compile(
    r'\bR\.?\s+v\.?\s+'
    r'[A-Z][\w\.\-\']+'
    r'(?:\s+(?:[A-Z][\w\.\-\']+|of|the|for|and|&|on|in|ex\s*p\.?|ex\s*parte|\([^)]*\)))*',
    re.UNICODE,
)

# 3. Re X / In re X (corporate/insolvency/probate — 4.09%)
RE_PATTERN = re.compile(
    r'\b(?:In\s+re|Re)\s+'
    r'[A-Z][\w\.\-\']+'
    r'(?:\s+(?:[A-Z][\w\.\-\']+|&|and|of|the|Co|Ltd|plc|Pte|Inc|Corp|Group|Holdings|\([^)]*\)|\(No\.?\s*\d+\)))*',
    re.UNICODE,
)

# 4. Ex parte X (NEW)
EX_PARTE_PATTERN = re.compile(
    r'\bex\s+parte\s+'
    r'[A-Z][\w\.\-\']+'
    r'(?:\s+(?:[A-Z][\w\.\-\']+|of|the|and|&|\([^)]*\)))*',
    re.IGNORECASE | re.UNICODE,
)

# 5. In the Matter of X (NEW)
IN_MATTER_PATTERN = re.compile(
    r'\bIn\s+the\s+Matter\s+of\s+'
    r'(?:an?\s+)?'
    r'[A-Z][\w\.\-\']+'
    r'(?:\s+(?:[A-Z][\w\.\-\']+|of|the|by|between|and|&|Ltd|plc|Pte|\([^)]*\)))*',
    re.IGNORECASE | re.UNICODE,
)

# 6. Generic Party v Party — catch-all (82.52%)
# V4: more permissive than V3 — catches single-letter parties (R, S),
# lowercase prefixes (de, von, van, der, du, le, la), $ and . in tokens (S$1.99 Pte Ltd),
# and number tokens (1.99 in 'Lifestyle 1.99 Pte Ltd')
_PARTY_TOKEN = (
    r'(?:'
    r'[A-Z][\w\.\-\'\$]*'                          # capitalised token
    r'|de|von|der|van|du|le|la|of|the|and|&'           # lowercase particles
    r'|\d+(?:\.\d+)?'                                 # numbers
    r')'
)
_PARTY_NAME = (
    r'(?:'
    r'[A-Z][\w\.\-\'\$]*'                          # standard cap start
    r'|(?:de|von|der|van|du|le|la)\s+[A-Z][\w]*'      # lowercase prefix + cap
    r')'
    r'(?:\s+' + _PARTY_TOKEN + r'){0,15}'
)
V_PATTERN = re.compile(
    r'\b' + _PARTY_NAME + r'\s+v\.?\s+' + _PARTY_NAME,
    re.UNICODE,
)

# =====================================================================
# JUDGE NAMES — catch identifiable judge references
# =====================================================================
LORD_LADY_PATTERN = re.compile(
    r'\b(?:Lord|Lady)\s+[A-Z][\w\-\']+(?:\s+(?:of|the|[A-Z][\w\-\']+))*',
    re.UNICODE,
)
JUDGE_TITLE_PATTERN = re.compile(
    r'\b[A-Z][\w\-\']+(?:\s+[A-Z][\w\-\']+)?'
    r'\s+(?:LJ|JA|JC|CJ|JCA|LCJ|MR|VC|J|JJ|CJJ)\b'
    r'(?:\.|s)?',
)

# =====================================================================
# CITATION RESIDUES — catch leftovers after main citation scrub
# =====================================================================
# Bare year markers
BARE_YEAR_BRACKET = re.compile(r'\[\s*(?:18|19|20)\d{2}\s*\]')
BARE_YEAR_PAREN   = re.compile(r'\(\s*(?:18|19|20)\d{2}\s*\)')

# Volume + reporter + page (e.g. '1 KB 1', '2 AC 489', '3 WLR 12')
# These are reporter citations missing a year prefix and need their own pattern.
# The reporter token is the anchor — we strip the volume number too if present.
VOL_REPORTER_PAGE = re.compile(
    r'\b\d+\s+'
    r'(?:AC|WLR|QB|KB|Ch|All\s*ER|SLR(?:\(R\))?|MLJ|TLR|RPC|FSR|BCLC|HKLR|'
    r'CLR|LR|App\s*Cas|Lloyd\'?s\s*Rep|FCR|F\.\s*\d+d?)'
    r'\s+\d+',
    re.IGNORECASE,
)

# Bare reporter abbreviations as standalone tokens
BARE_REPORTERS = re.compile(
    r'\b(?:SLR(?:\(R\))?|SGCA(?:\(I\))?|SGHC(?:\([A-Z]\))?|SGHCF|SGHCR|SGHCI|SGHCA|SGCAI|'
    r'EWCA(?:\s*Civ|\s*Crim)?|EWHC(?:\s*\([A-Za-z]+\))?|'
    r'UKSC|UKHL|UKPC|HKLR|HKCFA|HKCA|HKCFI|'
    r'AC|WLR|QB|KB|Ch|All\s*ER|'
    r'MLJ|TLR|RPC|FSR|BCLC|Lloyd\'?s\s*Rep|App\s*Cas|App\s*Cas\.?)\b',
    re.IGNORECASE,
)

# =====================================================================
# PINPOINTS
# =====================================================================
PINPOINTS = re.compile(
    r'\bat\s+'
    r'(?:\[\s*\d+\s*\](?:\s*[\u2013\u2014\-]\s*\[\s*\d+\s*\])?'
    r'|paras?\s*(?:\d+(?:\s*[\u2013\u2014\-]\s*\d+)?)?'
    r'|pp?\s*\.?\s*\d+(?:\s*[\u2013\u2014\-]\s*\d+)?)',
    re.IGNORECASE,
)

print('Citation patterns compiled (V4 — extended set with permissive Party v Party):')
print('  - SG neutral citations + SG reporters')
print('  - Foreign reporters (with and without year prefix)')
print('  - NEW: Volume + reporter + page (1 KB 1, 2 AC 489)')
print('  - Case-name patterns: PP v X, R v X, Re X, NEW Ex parte X, NEW In the Matter of X, Party v Party')
print('  - Judge names: Lord/Lady X, Surname + LJ/JA/JC/CJ/etc.')
print('  - Citation residues: bare years, expanded bare reporter abbreviations')
print('  - Pinpoints')


Citation patterns compiled (V4 — extended set with permissive Party v Party):
  - SG neutral citations + SG reporters
  - Foreign reporters (with and without year prefix)
  - NEW: Volume + reporter + page (1 KB 1, 2 AC 489)
  - Case-name patterns: PP v X, R v X, Re X, NEW Ex parte X, NEW In the Matter of X, Party v Party
  - Judge names: Lord/Lady X, Surname + LJ/JA/JC/CJ/etc.
  - Citation residues: bare years, expanded bare reporter abbreviations
  - Pinpoints


## Helper functions

In [3]:
def normalize_ws(s):
    return re.sub(r'\s+', ' ', s).strip()

def find_anchor_window(paragraph, cited_case, words_each_side=WORDS_EACH_SIDE):
    """Find first cited-case mention in the paragraph, take ±N words around it."""
    if not isinstance(paragraph, str):
        return ''
    words = paragraph.split()
    if len(words) <= 2 * words_each_side:
        return paragraph

    mid = None
    if isinstance(cited_case, str) and cited_case.strip():
        cc_tokens = cited_case.split()
        if cc_tokens:
            first = cc_tokens[0].lower().strip(',.;:()[]"')
            for i, w in enumerate(words):
                if w.lower().strip(',.;:()[]"') == first:
                    mid = i
                    break
    if mid is None:
        mid = len(words) // 2

    start = max(0, mid - words_each_side)
    end = min(len(words), mid + words_each_side)
    return ' '.join(words[start:end])

def scrub(text, cited_case):
    """Remove citation references, case names, judge names, citation residues."""
    if not isinstance(text, str):
        return ''
    out = text

    # 1. Remove the explicit cited-case string
    if isinstance(cited_case, str) and cited_case.strip():
        cc = normalize_ws(cited_case)
        if 5 <= len(cc) <= 250:
            tokens = [re.escape(t) for t in cc.split()]
            flex = r'\s+'.join(tokens)
            try:
                out = re.sub(flex, '[CASE]', out, flags=re.IGNORECASE)
            except re.error:
                pass

    # 2. Strip structured citation patterns
    out = SG_NEUTRAL.sub('[CITATION]', out)
    out = SG_REPORTERS.sub('[CITATION]', out)
    out = FOREIGN.sub('[CITATION]', out)
    out = PAREN_REPORTERS.sub('[CITATION]', out)

    # 3. Strip vol+reporter+page (e.g. '1 KB 1', '2 AC 489') BEFORE bare reporters
    out = VOL_REPORTER_PAGE.sub('[CITATION]', out)

    # 4. Strip case-name patterns (specific FIRST, generic LAST)
    out = PP_V_PATTERN.sub('[CASE]', out)
    out = R_V_PATTERN.sub('[CASE]', out)
    out = RE_PATTERN.sub('[CASE]', out)
    out = EX_PARTE_PATTERN.sub('[CASE]', out)
    out = IN_MATTER_PATTERN.sub('[CASE]', out)
    out = V_PATTERN.sub('[CASE]', out)

    # 5. Strip judge names
    out = LORD_LADY_PATTERN.sub('[JUDGE]', out)
    out = JUDGE_TITLE_PATTERN.sub('[JUDGE]', out)

    # 6. Strip citation residues
    out = BARE_YEAR_BRACKET.sub('[YEAR]', out)
    out = BARE_YEAR_PAREN.sub('[YEAR]', out)
    out = BARE_REPORTERS.sub('[REPORTER]', out)
    # Strip page numbers that immediately follow a [REPORTER] placeholder
    # (e.g. '[REPORTER] 489' → '[REPORTER]')
    out = re.sub(r'\[REPORTER\]\s+\d+(?:\s*[\u2013\u2014\-]\s*\d+)?', '[REPORTER]', out)

    # 7. Strip pinpoints
    out = PINPOINTS.sub('', out)

    # 8. Collapse repeated placeholders & whitespace
    out = re.sub(r'(\[CITATION\]\s*){2,}', '[CITATION] ', out)
    out = re.sub(r'(\[CASE\]\s*){2,}', '[CASE] ', out)
    out = re.sub(r'(\[JUDGE\]\s*){2,}', '[JUDGE] ', out)
    out = re.sub(r'(\[YEAR\]\s*){2,}', '[YEAR] ', out)
    out = re.sub(r'(\[REPORTER\]\s*){2,}', '[REPORTER] ', out)
    out = normalize_ws(out)
    return out

def build_scrubbed_window(paragraph, cited_case):
    """Combined pipeline: window first, then scrub."""
    windowed = find_anchor_window(paragraph, cited_case)
    return scrub(windowed, cited_case)

print('Helpers defined (V3).')

Helpers defined (V3).


## Smoke test on 5 rows

Eyeball the before/after to confirm scrubbing is sensible before applying to all 100,890 rows.

In [4]:
sample = pd.read_csv(INPUT_CSV,
                     usecols=['Cited Case', 'Paragraph', 'Key Principles Illustrated'],
                     dtype=str,
                     nrows=200)

# Take 5 spread across the sample so we see variety
for i in [3, 50, 100, 150, 199]:
    if i >= len(sample):
        continue
    row = sample.iloc[i]
    cc = row['Cited Case']
    para = row['Paragraph']
    if not isinstance(para, str):
        continue

    print(f"=== Sample row {i} ===")
    print(f"Cited Case: {(str(cc) or '')[:120]}")
    print(f"Paragraph length: {len(para.split())} words")

    out = build_scrubbed_window(para, cc)
    print(f"Scrubbed-window length: {len(out.split())} words")
    print(f"Scrubbed text (first 600 chars):\n{out[:600]}")
    print(f"\nFor reference, Key Principle ({len(str(row['Key Principles Illustrated']).split())} words):\n{str(row['Key Principles Illustrated'])[:300]}")
    print()

=== Sample row 3 ===
Cited Case: Black v Yates [1992]
Paragraph length: 1272 words
Scrubbed-window length: 357 words
Scrubbed text (first 600 chars):
parties`, and stated that the plea applies `to every point which properly belonged to the subject of litigation, and which the parties, exercising reasonable diligence, might have brought forward at the time `. [My emphasis.] [CASE] , properly understood, held that the plaintiffs who were barred from bringing their later action were indeed able to be regarded as parties during the earlier, somewhat convoluted, course of the litigation; their new claim was in reality, if sound, a defence available in a previous action in which they had clearly been interested although not formally a party. The 

For reference, Key Principle (117 words):
The doctrine of abuse of process ought only to be applied when the facts are such as to amount to an abuse: otherwise there is a danger of a party being shut out from bringing forward a genuine subject of l

## Apply to full dataset (chunked to keep memory reasonable)

100,890 rows × ~1,100 words each × multiple regex passes = real work but not slow. Chunked at 5,000 rows to keep memory low. Should take 5–15 minutes.

In [5]:
import time
from datetime import datetime

CHUNK_SIZE = 5000
first_chunk = True
total_rows = 0
start = time.time()

for chunk_idx, chunk in enumerate(pd.read_csv(INPUT_CSV, dtype=str, chunksize=CHUNK_SIZE)):
    chunk['Paragraph_Scrubbed_Window'] = [
        build_scrubbed_window(p, c)
        for p, c in zip(chunk['Paragraph'], chunk['Cited Case'])
    ]
    chunk.to_csv(OUTPUT_CSV, mode='w' if first_chunk else 'a',
                 header=first_chunk, index=False)
    first_chunk = False
    total_rows += len(chunk)
    elapsed = time.time() - start
    print(f"[{datetime.now().strftime('%H:%M:%S')}] chunk {chunk_idx+1}: "
          f"{total_rows:,} rows done, {elapsed:.0f}s elapsed", flush=True)

print(f"\n✅ Done. {total_rows:,} rows written to {OUTPUT_CSV}")
print(f"   Total time: {(time.time()-start)/60:.1f} min")

[15:13:21] chunk 1: 5,000 rows done, 3s elapsed
[15:13:24] chunk 2: 10,000 rows done, 6s elapsed
[15:13:27] chunk 3: 15,000 rows done, 10s elapsed
[15:13:31] chunk 4: 20,000 rows done, 13s elapsed
[15:13:34] chunk 5: 25,000 rows done, 16s elapsed
[15:13:38] chunk 6: 30,000 rows done, 20s elapsed
[15:13:41] chunk 7: 35,000 rows done, 23s elapsed
[15:13:45] chunk 8: 40,000 rows done, 27s elapsed
[15:13:48] chunk 9: 45,000 rows done, 30s elapsed
[15:13:52] chunk 10: 50,000 rows done, 34s elapsed
[15:13:55] chunk 11: 55,000 rows done, 37s elapsed
[15:13:59] chunk 12: 60,000 rows done, 41s elapsed
[15:14:02] chunk 13: 65,000 rows done, 44s elapsed
[15:14:06] chunk 14: 70,000 rows done, 48s elapsed
[15:14:09] chunk 15: 75,000 rows done, 51s elapsed
[15:14:13] chunk 16: 80,000 rows done, 55s elapsed
[15:14:16] chunk 17: 85,000 rows done, 58s elapsed
[15:14:20] chunk 18: 90,000 rows done, 62s elapsed
[15:14:23] chunk 19: 95,000 rows done, 65s elapsed
[15:14:27] chunk 20: 100,000 rows done, 69s

## Verify the output

In [6]:
# Quick sanity check on the produced file - chunked load
lengths = []
missing = 0
for chunk in pd.read_csv(OUTPUT_CSV,
                          usecols=['Paragraph_Scrubbed_Window'],
                          dtype=str,
                          chunksize=10000):
    chunk['n_words'] = chunk['Paragraph_Scrubbed_Window'].fillna('').str.split().str.len()
    lengths.extend(chunk['n_words'].tolist())
    missing += chunk['Paragraph_Scrubbed_Window'].isna().sum()

import numpy as np
arr = np.array(lengths)

print(f"Rows: {len(arr):,}")
print(f"Missing scrubbed-window: {missing}")
print(f"\nPARAGRAPH_SCRUBBED_WINDOW length (words):")
print(f"  mean:   {arr.mean():.0f}")
print(f"  median: {np.median(arr):.0f}")
print(f"  min:    {arr.min()}")
print(f"  max:    {arr.max():,}")
print(f"  95th:   {np.percentile(arr, 95):.0f}")
print(f"  99th:   {np.percentile(arr, 99):.0f}")

# Approx tokens for BERT (×1.3 for subword)
print(f"\nApprox tokens (×1.3): mean ≈ {arr.mean()*1.3:.0f}, 95th ≈ {np.percentile(arr, 95)*1.3:.0f}")

# How many will fit in 512 tokens together with ~60-token fact summary?
# Available budget for paragraph ≈ 512 - 60 - 5 (special tokens) = 447 tokens ≈ 344 words
fits_pct = (arr <= 344).mean() * 100
print(f"\n% of scrubbed windows that fit in 512-token encoder with fact: {fits_pct:.1f}%")

Rows: 100,890
Missing scrubbed-window: 0

PARAGRAPH_SCRUBBED_WINDOW length (words):
  mean:   352
  median: 372
  min:    12
  max:    400
  95th:   391
  99th:   395

Approx tokens (×1.3): mean ≈ 457, 95th ≈ 508

% of scrubbed windows that fit in 512-token encoder with fact: 20.2%


## Final spot-check: 3 random rows from the saved output

In [7]:
check = pd.read_csv(OUTPUT_CSV,
                    usecols=['Cited Case', 'Key Principles Illustrated', 'Paragraph_Scrubbed_Window'],
                    dtype=str,
                    nrows=5000)
for i in check.sample(3, random_state=7).index:
    row = check.loc[i]
    print(f"=== Row {i} ===")
    print(f"Cited Case: {str(row['Cited Case'])[:120]}")
    print(f"\nKey Principle: {str(row['Key Principles Illustrated'])[:300]}")
    print(f"\nScrubbed Window (first 800 chars):\n{str(row['Paragraph_Scrubbed_Window'])[:800]}")
    print()

=== Row 3406 ===
Cited Case: PP v Azman bin Abdullah [1998]

Key Principle: An appellate court will be reluctant to overturn the trial judge's findings of fact unless they were clearly reached against the weight of the evidence or they were plainly wrong

Scrubbed Window (first 800 chars):
from a fall as a result of a collision. In addition, I also found that the judge had good reasons to find that Chew was evasive and not a credible witness. He vacillated on crucial issues such as whether or not there was a collision, claiming in his earlier statement that he did not know and then becoming very sure that there was no collision when he was on the stand. He gave three different versions of how close he was to Ahmad`s bicycle when he first spotted it - in statement P29 he said that he was four feet away; in statement P30 he said that he was more than six feet but less than seven feet away; and on the stand he said that he was ten feet away. In addition, although he claimed that there was